In [16]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    filename="logs/mgmt_operations.as_244.log",
    encoding="utf-8",
)

LOG: logging.Logger = logging.getLogger(__name__)

In [17]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage
from epicsarchiver import ArchiverAppliance

In [18]:
from concurrent.futures import ThreadPoolExecutor

In [19]:

import random

def delete_multiple_pvs(archivers, pv_list):
    
    def delete_pv(args):
        archiver, pv = args
        pv_status = archiver.get_archiving_status(pv)
        if pv_status != ArchivingStatus.BeingArchived:
            LOG.info(f"PV {pv} is not being archived, skipping")
        LOG.info(f"Deleting PV {pv}, first pausing")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pv}:{pause_result}")
        delete_result = archiver.delete_pv(pv)
        LOG.info(f"Delete result: {pv}:{delete_result}")
        pv_status = archiver.get_archiving_status(pv)
        LOG.info(f"PV status: {pv}: {pv_status}")
    
    with ThreadPoolExecutor() as executor:
        executor.map(delete_pv, [(ArchiverAppliance(random.choice(archivers).hostname),pv) for pv in pv_list])

In [20]:
def pause_multiple_pvs(archivers, pv_list):
    def pause_pv(args):
        archiver, pv = args
        LOG.info(f"Pausing PV: {pv}")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pv}: {pause_result}")
    with ThreadPoolExecutor() as executor:
        executor.map(pause_pv, [(ArchiverAppliance(random.choice(archivers).hostname),pv) for pv in pv_list])
        

In [21]:
def multiple_change_archiving_rate(archivers: list[ArchiverAppliance], input_pv_rate: list[tuple[str, str]]):
    def change_rate(args):
        archiver, pv, rate = args
        if rate == "14Hz":
            LOG.info(f"Setting {pv} to 14Hz")
            archiver.update_pv(pv, 0.07)
        elif rate == "1Hz":
            LOG.info(f"Setting {pv} to 1Hz")
            archiver.update_pv(pv, 1.0)
        else:
            LOG.info(f"Doing nothing for {pv} ")
    inputs = [(ArchiverAppliance(random.choice(archivers).hostname), pv, rate) for pv, rate in input_pv_rate]
    with ThreadPoolExecutor() as executor:
        executor.map(change_rate, inputs )

In [22]:
archiver_linac_tns = [ArchiverAppliance(f"archiver-linac-0{i}.tn.esss.lu.se") for i in range(2, 9)]

In [23]:
from pathlib import Path
def get_files(folder, ending):
    return [f for f in Path(folder).iterdir() if f.is_file() and f.name.endswith(ending)] 

## Updates

In [24]:
import csv

def read_to_update_file(filename):
    file_data = csv.reader(open(filename))
    split_data = [row[0].split() for row in file_data]
    return [(row[0], row[1] if len(row) == 2 else "14Hz") for row in split_data if row]

def read_update_files(folder):
    files = get_files(folder, "ToUpdate_PVs.archiver")
    return {f.name: read_to_update_file(f) for f in files}


In [25]:
def change_from_files(archivers, files_data):
    for updates, updates_changes in files_data.items():
        LOG.info(f"updates: {updates}")
        multiple_change_archiving_rate(archivers, updates_changes)

In [26]:

update_spk_data = read_update_files("AS-244/archiver_files")


In [27]:
update_spk_data

{'spk_110cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-110Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_020cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-020Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_120cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-120Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_010cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-010Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_070cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-070Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_040cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-040Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_080cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-080Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_090cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-090Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_050cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-050Crm:SC-FSM-003:CDS_Cryo_OK',
   '14Hz')],
 'spk_060cdl_cryo_plc_010_ToUpdate_PVs.archiver': [('Spk-060Crm:SC-FSM-003:CDS_Cry

In [28]:
change_from_files(archiver_linac_tns, update_spk_data)

## Deletes

In [33]:

def read_to_delete_file(filename: Path):
    result = []
    with open(filename, 'r') as file:
        for line in file.readlines():
            if not line.startswith("#"):
                result.append(line.strip().split()[0])
    return result

def read_delete_files(folder):
    files =  get_files(folder, "Verify_PVs.archiver")
    return {f.name: read_to_delete_file(f) for f in files}

In [34]:
def delete_all(archivers, delete_data):
    for _, file_data in delete_data.items():
        delete_multiple_pvs(archivers, file_data)

In [35]:
delete_spk_data = read_delete_files("AS-244/archiver_files")

In [36]:
delete_spk_data

{'spk_010cdl_cryo_plc_010_Verify_PVs.archiver': ['Spk-010Crm:SC-FSM-101:WU_4K_lvl_dec',
  'Spk-010Crm:SC-FSM-101:WU_300K_lvl_dec',
  'Spk-010Crm:SC-FSM-101:PerT310',
  'Spk-010Crm:SC-FSM-101:PerT505',
  'Spk-010Crm:SC-FSM-101:PerT700',
  'Spk-010Crm:SC-FSM-101:PerT712',
  'Spk-010Crm:SC-FSM-101:PerT800',
  'Spk-010Crm:SC-FSM-101:PerRes9',
  'Spk-010Crm:SC-FSM-101:PerRes10',
  'Spk-010CDL:SC-FSM-300:OM_Undefined',
  'Spk-010CDL:SC-FSM-300:OM_Stopped',
  'Spk-010CDL:SC-FSM-300:OM_Purging',
  'Spk-010CDL:SC-FSM-300:OM_Stand_by_4K',
  'Spk-010CDL:SC-FSM-300:OM_Stand_by_2K',
  'Spk-010CDL:SC-FSM-300:OM_Nominal_2K_RF',
  'Spk-010CDL:SC-FSM-300:OM_Starting',
  'Spk-010CDL:SC-FSM-300:OM_Ready_for_RF',
  'Spk-010CDL:SC-FSM-300:OM_RF_OFF',
  'Spk-010CDL:SC-FSM-300:OM_S_CD_300-4K',
  'Spk-010CDL:SC-FSM-300:OM_S_CD_4-2K',
  'Spk-010CDL:SC-FSM-300:OM_S_WU_4-300K',
  'Spk-010CDL:SC-FSM-300:OM_S_WU_2-4K',
  'Spk-010CDL:SC-FSM-300:Rdy_for_pumpdown',
  'Spk-010CDL:SC-FSM-300:Pumpdown_started',
  'Spk-0

In [37]:
delete_all(archiver_linac_tns, delete_spk_data)